# Historical Weather Data Analysis 🌦️

**Dataset:** `weather_data.csv`

## Research questions
1. What is the distribution of temperature?
2. How is humidity distributed?
3. What is the distribution of wind speed?
4. Is there a relationship between temperature and humidity?
5. How does apparent temperature compare with actual temperature?

**Dataset note:** This file has no country/place column. It contains weather observations with a date field, so country comparisons are not possible. Multiple observations can share a calendar date; therefore, a repeated date is not automatically a duplicate.

## 1. Load libraries and data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df = pd.read_csv("weather_data.csv")
print("Dataset loaded.")

In [ ]:
df.head()

The first five records show the structure and example values in the dataset.

In [ ]:
df.info()

`info()` shows the column names, data types, non-null counts and memory use.

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

The dataset contains 96,453 rows and 14 columns. It has weather measurements, a date field and calendar fields, but no location/country column.

## 2. Basic analysis and date preparation

In [ ]:
df["Formatted Date"] = pd.to_datetime(df["Formatted Date"], errors="coerce")
print("First date:", df["Formatted Date"].min())
print("Last date:", df["Formatted Date"].max())
print("Unique calendar dates:", df["Formatted Date"].dt.date.nunique())
print("Place/country column available:", any(c.lower() in ["country", "place", "location", "city"] for c in df.columns))

The date field is converted to pandas datetime. There is no place or country field, so the number of countries/places cannot be calculated.

In [ ]:
df.describe(include="all").T

Descriptive statistics summarize numeric columns and show counts/frequencies for text columns.

### Select columns for this analysis

In [ ]:
selected_columns = [
    "Formatted Date", "Summary", "Precip Type",
    "Temperature (C)", "Apparent Temperature (C)",
    "Humidity", "Wind Speed (km/h)", "Visibility (km)",
    "Pressure (millibars)"
]
weather = df[selected_columns].copy()
weather.head()

Only columns relevant to the research questions and useful weather context are retained for the analysis.

## 3. Missing values

In [ ]:
missing = weather.isna().sum().sort_values(ascending=False)
display(missing.to_frame("Missing values"))

plt.figure(figsize=(10, 4))
missing.plot(kind="bar")
plt.title("Missing Values by Column")
plt.xlabel("Column")
plt.ylabel("Number of missing values")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Only `Precip Type` has missing values (517). It is categorical, so missing entries are labelled `Unknown` rather than filled with a numeric median or previous observation. The analysis columns used in the five research questions have no missing values.

In [ ]:
weather["Precip Type"] = weather["Precip Type"].fillna("Unknown")
print(weather.isna().sum())

After filling the missing precipitation labels with `Unknown`, the selected analysis table has no missing values.

## 4. Repeated values

In [ ]:
duplicate_rows = weather.duplicated().sum()
print("Completely duplicated rows:", duplicate_rows)

# A date can appear many times because the file contains multiple observations per day.
rows_per_date = weather.groupby(weather["Formatted Date"].dt.date).size()
print("Dates with more than one observation:", (rows_per_date > 1).sum())
print("Maximum observations on one date:", rows_per_date.max())

There are 29 completely duplicated records in the original dataset. Repeated dates are expected because the dataset contains multiple observations per calendar day; they must not be treated as duplicate records by date alone.

In [ ]:
weather = weather.drop_duplicates().copy()
print("Rows after removing exact duplicates:", len(weather))
print("Remaining exact duplicates:", weather.duplicated().sum())

Only fully identical rows are removed. This preserves different weather observations recorded on the same date.

## 5. Outliers

In [ ]:
numeric_weather = ["Temperature (C)", "Wind Speed (km/h)", "Humidity"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_weather):
    sns.boxplot(y=weather[col], ax=ax)
    ax.set_title(f"Box Plot: {col}")
plt.tight_layout()
plt.show()

Box plots help identify unusually low or high observations. An outlier is not automatically an error in weather data.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_weather):
    sns.histplot(weather[col], kde=True, ax=ax)
    ax.set_title(f"Histogram: {col}")
    ax.set_xlabel(col)
plt.tight_layout()
plt.show()

Histograms show the shape of each variable's distribution and make skewness or extreme values easier to inspect.

In [ ]:
def iqr_outlier_report(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((series < lower) | (series > upper)).sum()
    return pd.Series({"Q1": q1, "Q3": q3, "IQR": iqr,
                      "Lower fence": lower, "Upper fence": upper,
                      "Outlier count": count})

outlier_results = pd.DataFrame({
    col: iqr_outlier_report(weather[col]) for col in
    ["Temperature (C)", "Wind Speed (km/h)", "Humidity"]
}).T
outlier_results

The IQR rule flags values outside 1.5 × IQR from the quartiles. These observations are retained unless they can be confirmed as recording errors: extreme temperatures or wind speeds may represent genuine weather events. Humidity is naturally bounded between 0 and 1 in this dataset.

## 6. Univariate analysis

In [ ]:
univariate_columns = [
    "Temperature (C)", "Apparent Temperature (C)",
    "Humidity", "Wind Speed (km/h)",
    "Visibility (km)", "Pressure (millibars)"
]
for col in univariate_columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(weather[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()
    print(f"{col}: mean={weather[col].mean():.2f}, median={weather[col].median():.2f}, std={weather[col].std():.2f}")

Each histogram shows one weather variable. The printed mean, median and standard deviation describe its typical value and spread.

## 7. Bivariate and time analysis

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=weather, x="Temperature (C)", y="Humidity", alpha=0.25, s=12)
plt.title("Temperature vs Humidity")
plt.xlabel("Temperature (°C)")
plt.ylabel("Humidity")
plt.tight_layout()
plt.show()

The scatter plot shows whether humidity tends to rise or fall as temperature changes.

In [ ]:
corr_cols = [
    "Temperature (C)", "Apparent Temperature (C)", "Humidity",
    "Wind Speed (km/h)", "Visibility (km)", "Pressure (millibars)"
]
plt.figure(figsize=(9, 6))
sns.heatmap(weather[corr_cols].corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Weather Variables Correlation Heatmap")
plt.tight_layout()
plt.show()

The heatmap displays pairwise linear correlations. Values near +1 indicate a strong positive relationship, values near -1 a strong negative relationship, and values near 0 a weak linear relationship.

In [ ]:
daily_temp = weather.groupby(weather["Formatted Date"].dt.date)["Temperature (C)"].mean()
plt.figure(figsize=(13, 5))
daily_temp.plot()
plt.title("Average Temperature Over Time")
plt.xlabel("Date")
plt.ylabel("Average Temperature (°C)")
plt.tight_layout()
plt.show()

This line chart shows the daily average temperature over the available date range. It uses daily averages because multiple observations can occur on one date.

In [ ]:
weather["Year"] = weather["Formatted Date"].dt.year
yearly_temp = weather.groupby("Year")["Temperature (C)"].mean()
plt.figure(figsize=(9, 4))
yearly_temp.plot(kind="bar")
plt.title("Average Temperature by Year")
plt.xlabel("Year")
plt.ylabel("Average Temperature (°C)")
plt.tight_layout()
plt.show()

A country comparison is not possible because the dataset contains no country/place field. This bar chart provides a valid alternative by comparing yearly average temperature.

## 8. Answer the five research questions

In [ ]:
# Q1: Temperature distribution
display(weather["Temperature (C)"].describe().to_frame("Temperature (°C)"))
plt.figure(figsize=(8, 4))
sns.histplot(weather["Temperature (C)"], kde=True)
plt.title("Q1: Distribution of Temperature")
plt.xlabel("Temperature (°C)")
plt.ylabel("Frequency")
plt.show()
print("Answer: The mean temperature is", round(weather["Temperature (C)"].mean(), 2),
      "°C and the median is", round(weather["Temperature (C)"].median(), 2), "°C.")

In [ ]:
# Q2: Humidity distribution
display(weather["Humidity"].describe().to_frame("Humidity"))
plt.figure(figsize=(8, 4))
sns.histplot(weather["Humidity"], kde=True)
plt.title("Q2: Distribution of Humidity")
plt.xlabel("Humidity (0 to 1)")
plt.ylabel("Frequency")
plt.show()
print("Answer: Mean humidity =", round(weather["Humidity"].mean(), 3),
      "and median humidity =", round(weather["Humidity"].median(), 3))

In [ ]:
# Q3: Wind speed distribution
display(weather["Wind Speed (km/h)"].describe().to_frame("Wind speed (km/h)"))
plt.figure(figsize=(8, 4))
sns.histplot(weather["Wind Speed (km/h)"], kde=True)
plt.title("Q3: Distribution of Wind Speed")
plt.xlabel("Wind Speed (km/h)")
plt.ylabel("Frequency")
plt.show()
print("Answer: Mean wind speed =", round(weather["Wind Speed (km/h)"].mean(), 2),
      "km/h; median =", round(weather["Wind Speed (km/h)"].median(), 2), "km/h.")

In [ ]:
# Q4: Temperature and humidity relationship
r = weather["Temperature (C)"].corr(weather["Humidity"])
sns.regplot(data=weather.sample(min(5000, len(weather)), random_state=42),
            x="Temperature (C)", y="Humidity",
            scatter_kws={"alpha": 0.2, "s": 10}, line_kws={"color": "red"})
plt.title("Q4: Temperature vs Humidity")
plt.xlabel("Temperature (°C)")
plt.ylabel("Humidity")
plt.show()
print(f"Answer: Pearson correlation = {r:.3f}. This indicates the direction and strength of their linear relationship.")

In [ ]:
# Q5: Actual vs apparent temperature
comparison = weather[["Temperature (C)", "Apparent Temperature (C)"]].agg(["mean", "median", "std"])
display(comparison)
sample = weather.sample(min(5000, len(weather)), random_state=42)
plt.figure(figsize=(7, 6))
sns.scatterplot(data=sample, x="Temperature (C)", y="Apparent Temperature (C)", alpha=0.25, s=12)
low = min(sample["Temperature (C)"].min(), sample["Apparent Temperature (C)"].min())
high = max(sample["Temperature (C)"].max(), sample["Apparent Temperature (C)"].max())
plt.plot([low, high], [low, high], linestyle="--", color="red")
plt.title("Q5: Actual vs Apparent Temperature")
plt.xlabel("Temperature (°C)")
plt.ylabel("Apparent Temperature (°C)")
plt.tight_layout()
plt.show()
difference = weather["Apparent Temperature (C)"] - weather["Temperature (C)"]
print("Answer: Average apparent-minus-actual temperature difference =",
      round(difference.mean(), 2), "°C.")

## 9. Conclusion

In [ ]:
print(
    f"1. The dataset contains {len(weather):,} observations after exact duplicate removal.\n"
    f"2. It covers {weather['Formatted Date'].min().date()} to {weather['Formatted Date'].max().date()}.\n"
    f"3. Average temperature is {weather['Temperature (C)'].mean():.2f} °C.\n"
    f"4. Average humidity is {weather['Humidity'].mean():.2f} (0–1 scale).\n"
    f"5. Average wind speed is {weather['Wind Speed (km/h)'].mean():.2f} km/h.\n"
    f"6. Temperature and humidity correlation is {weather['Temperature (C)'].corr(weather['Humidity']):.2f}. "
    "The dataset has no location column, so geographic comparisons are unavailable."
)

## Notes for submission
- The dataset does not contain a country/place column; the dashboard therefore uses a date range filter and does not offer a country filter.
- The file contains repeated calendar dates because there are multiple observations per day. Only fully identical rows are removed.
- No outliers are removed automatically; unusual weather values require context before being treated as errors.